# Self-supervised YouTube-ASL Skeleton BERT

This notebook clones the code from GitHub and keeps keypoint data only in Colab's temporary `/content` storage. It saves checkpoints to Google Drive when Drive mounting works; otherwise it downloads a compact resume bundle after every completed shard. Choose **Runtime → Change runtime type → GPU** first.

This run defaults to `MODE = 'full'`. Full mode downloads and trains one shard at a time, persists resumable checkpoints, and removes each local data shard after it finishes.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## 1. Configure checkpoint persistence and clone GitHub

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

DRIVE_MOUNTED = False
try:
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as error:  # noqa: BLE001 -- Colab exposes several mount error types
    print('Drive mount unavailable; using browser-download persistence:', error)

REPO_URL = 'https://github.com/ss-sebastian/youtube-asl-skeleton-bert.git'
PROJECT = Path('/content/youtube-asl-skeleton-bert')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
print('Project ready:', PROJECT)

## 2. Select pilot or resumable full training

In [ ]:
MODE = 'full'  # full training over all official shards
PILOT_TRAIN_CLIPS = 512
PILOT_VAL_CLIPS = 128
FULL_SHARDS = list(range(1, 11))
BATCH_SIZE = 16
MAX_FRAMES = 256

LOCAL_ROOT = Path('/content/youtube_asl_data')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
PERSIST_ROOT = (
    Path('/content/drive/MyDrive/sign_semantics_youtube_asl')
    if DRIVE_MOUNTED
    else Path('/content/sign_semantics_youtube_asl')
)
RUN_ROOT = PERSIST_ROOT / MODE
CHECKPOINT_ROOT = RUN_ROOT / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESUME_BUNDLE = Path('/content/youtube_asl_full_resume.zip')
if not DRIVE_MOUNTED and RESUME_BUNDLE.exists():
    with zipfile.ZipFile(RESUME_BUNDLE) as handle:
        handle.extractall(RUN_ROOT)
    print('Restored prior full-run state from:', RESUME_BUNDLE)
print('Data (temporary):', LOCAL_ROOT)
print('Models:', CHECKPOINT_ROOT)
print('Persistence:', 'Google Drive' if DRIVE_MOUNTED else 'download bundle after every shard')

## 3. Official dataset URLs and train/dev annotations

In [ ]:
SHARDS = {
 1: ('3dc57bf4-c5fb-491c-8ab2-a9215e0e2fe5', '75223dd5e7e9b6ccb9f34c5792fc1e6d'),
 2: ('bfa38e27-bd46-48ae-bf9a-eb1eafaabb95', 'c891a51901ca17fa6f42529e5657df67'),
 3: ('0c59bda5-908d-4194-93bf-13e13de2ef10', '0713086c142d52c31cff1b4be9f4f82a'),
 4: ('1a4ace36-ed9a-4bfb-a80f-483c0463e02d', '890dd2fa779b56cd90c6ae39fa67faef'),
 5: ('6b405702-9f74-4729-8202-55eca36adaea', 'e935642e8bb5f9b594af74a8ba75d97f'),
 6: ('3b4ef094-bc95-4efb-b8e0-3bc4f63e57b0', '403958646966402245f69f9d473c4346'),
 7: ('74e99da7-c580-4fdf-8363-8024d7a7adf1', '786fe0067d1e4d8665b1ddbb8628a17c'),
 8: ('43a9146e-0abf-47ec-a3b3-90c01a4d9380', 'b6bcc2e8517c2dcf8347347bbd74800c'),
 9: ('05385788-d459-4f35-92b4-4908b9d86de6', '9d52caab1aa2db4218188819485f92ab'),
10: ('d96b874e-72fa-4830-b2f0-a072bb6be31d', '0019b603f9ebd7594fce8fae8dc65167'),
}
BASE = 'https://lindat.mff.cuni.cz/repository/server/api/core/bitstreams'
TRAIN_ANNOTATION_URL = f'{BASE}/f8460818-3605-4f05-9832-90ddc68f22e6/content'
DEV_ANNOTATION_URL = f'{BASE}/d5d23d31-c93a-4752-8e5c-e20548f13da0/content'
TRAIN_ANNOTATIONS = LOCAL_ROOT / 'YT.translations.train.json'
DEV_ANNOTATIONS = LOCAL_ROOT / 'YT.translations.dev.json'
for url, target in [(TRAIN_ANNOTATION_URL, TRAIN_ANNOTATIONS), (DEV_ANNOTATION_URL, DEV_ANNOTATIONS)]:
    if not target.exists():
        subprocess.run(['wget', '-q', '--show-progress', '-O', str(target), url], check=True)
print('Annotations:', TRAIN_ANNOTATIONS.stat().st_size, DEV_ANNOTATIONS.stat().st_size)

## 4. Train

Pilot mode reads limited entries through HTTP range requests. Full mode needs roughly 45 GB free for one compressed shard; it never extracts the ZIP. `completed_shards.json` and `last.pt` make the full run resumable across Colab sessions. If Drive mounting is unavailable, upload the latest downloaded resume bundle before rerunning and rename it `/content/youtube_asl_full_resume.zip`; the configuration cell restores it automatically.

In [ ]:
import hashlib
import json


def shard_url(number):
    return f'{BASE}/{SHARDS[number][0]}/content'

def md5(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.md5()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_config(archive, epochs, pilot=False):
    config = json.loads((PROJECT / 'configs/pretrain.json').read_text())
    config['data'].update({
        'train_archive': str(archive),
        'val_archive': str(archive),
        'train_annotations': str(TRAIN_ANNOTATIONS),
        'val_annotations': str(DEV_ANNOTATIONS),
        'max_frames': MAX_FRAMES,
        'num_workers': 0 if pilot else 2,
    })
    if pilot:
        config['data']['limit_train_clips'] = PILOT_TRAIN_CLIPS
        config['data']['limit_val_clips'] = PILOT_VAL_CLIPS
    config['training'].update({
        'output_dir': str(CHECKPOINT_ROOT),
        'batch_size': BATCH_SIZE,
        'epochs': epochs,
        'amp': True,
    })
    path = Path('/content/colab_pretrain.json')
    path.write_text(json.dumps(config, indent=2))
    return path

def train_once(config_path, resume=None):
    command = ['sign-pretrain', '--config', str(config_path)]
    if resume is not None and resume.exists():
        command += ['--resume', str(resume)]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

def download_resume_bundle(state_path, shard):
    if DRIVE_MOUNTED:
        return
    bundle = Path(f'/content/youtube_asl_full_resume_after_shard_{shard:02d}.zip')
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as handle:
        handle.write(state_path, 'completed_shards.json')
        for name in ('last.pt', 'best.pt'):
            checkpoint = CHECKPOINT_ROOT / name
            if checkpoint.exists():
                handle.write(checkpoint, f'checkpoints/{name}')
    print('Downloading resumable checkpoint bundle:', bundle)
    files.download(str(bundle))

if MODE == 'pilot':
    config_path = write_config(shard_url(1), epochs=1, pilot=True)
    train_once(config_path)
else:
    state_path = RUN_ROOT / 'completed_shards.json'
    completed = json.loads(state_path.read_text()) if state_path.exists() else []
    for shard in FULL_SHARDS:
        if shard in completed:
            print(f'Shard {shard} already completed; skipping.')
            continue
        free_gb = shutil.disk_usage('/content').free / 1024**3
        if free_gb < 45:
            raise RuntimeError(f'Need at least 45 GB free; only {free_gb:.1f} GB available.')
        archive = LOCAL_ROOT / f'raw_keypoints_{shard}.zip'
        subprocess.run(['wget', '-c', '--show-progress', '-O', str(archive), shard_url(shard)], check=True)
        actual_md5 = md5(archive)
        if actual_md5 != SHARDS[shard][1]:
            raise RuntimeError(f'MD5 mismatch for shard {shard}: {actual_md5}')
        config_path = write_config(archive, epochs=len(completed) + 1)
        resume = CHECKPOINT_ROOT / 'last.pt'
        train_once(config_path, resume if completed else None)
        completed.append(shard)
        state_path.write_text(json.dumps(completed))
        download_resume_bundle(state_path, shard)
        archive.unlink()
        print(f'Completed shard {shard}; local data removed.')

## 5. Verify the persistent model

In [ ]:
for path in sorted(CHECKPOINT_ROOT.glob('*.pt')):
    print(path, f'{path.stat().st_size / 1024**2:.1f} MB')
assert (CHECKPOINT_ROOT / 'last.pt').exists(), 'Training did not write last.pt'
print('Persistent model:', CHECKPOINT_ROOT / 'last.pt')